# Exploratory Data Analysis — Football Players Detection

**Dataset:** `roboflow-jvuqo/football-players-detection-2frwp` v1 (CC BY 4.0)  
**Source:** [Roboflow Universe](https://universe.roboflow.com/roboflow-jvuqo/football-players-detection-2frwp/dataset/1)  
**Format:** YOLO (txt labels: `cls cx cy w h`, normalized 0-1)

## What this notebook covers
1. Path & structure sanity check
2. Image counts per split (train / valid / test)
3. Class distribution (instances per class per split)
4. Image-size distribution
5. Bounding-box size distribution (area, aspect ratio)
6. Annotations-per-image distribution
7. Visual samples (random + per-class)
8. Anomaly checks (empty labels, malformed boxes)
9. Conclusions & implications for training

## 0. Setup

In [ ]:
from __future__ import annotations

import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
from PIL import Image

sns.set_theme(style="whitegrid", context="notebook")
random.seed(42)
np.random.seed(42)

# Project root (assumes notebook is at notebooks/01_eda.ipynb)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_ROOT = PROJECT_ROOT / "data" / "processed" / "football-players"

print(f"Project root: {PROJECT_ROOT}")
print(f"Data root:    {DATA_ROOT}")
assert DATA_ROOT.exists(), f"Dataset folder not found: {DATA_ROOT}"

## 1. Path & structure sanity check

In [ ]:
# Load data.yaml
data_yaml = yaml.safe_load((DATA_ROOT / "data.yaml").read_text())
CLASS_NAMES: list[str] = data_yaml["names"]
NUM_CLASSES: int = data_yaml["nc"]

print(f"Classes ({NUM_CLASSES}): {CLASS_NAMES}")
print("\nExpected layout:")
for split in ["train", "valid", "test"]:
    images = DATA_ROOT / split / "images"
    labels = DATA_ROOT / split / "labels"
    print(f"  {split}: images={images.exists()} labels={labels.exists()}")

## 2. Image counts per split

In [ ]:
SPLITS = ["train", "valid", "test"]
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}


def list_images(split: str) -> list[Path]:
    return [p for p in (DATA_ROOT / split / "images").iterdir() if p.suffix.lower() in IMAGE_EXTS]


def list_labels(split: str) -> list[Path]:
    return list((DATA_ROOT / split / "labels").glob("*.txt"))


split_counts = {s: {"images": len(list_images(s)), "labels": len(list_labels(s))} for s in SPLITS}
split_df = pd.DataFrame(split_counts).T
split_df["diff"] = split_df["images"] - split_df["labels"]
print(split_df)

fig, ax = plt.subplots(1, 1, figsize=(7, 4))
split_df[["images", "labels"]].plot(kind="bar", ax=ax)
ax.set_title("Images vs labels per split")
ax.set_ylabel("count")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

**Watch out:** if `diff != 0` for a split, some images don't have corresponding label files (or vice versa). For YOLO this is usually OK (an empty `.txt` means a negative sample), but worth knowing.

## 3. Class distribution

We parse every `.txt` and count instances of each class per split.

In [ ]:
def parse_yolo_label(label_file: Path) -> list[tuple[int, float, float, float, float]]:
    """Return list of (cls_id, cx, cy, w, h). All coords normalized in [0, 1]."""
    rows = []
    if not label_file.exists():
        return rows
    for line in label_file.read_text().strip().splitlines():
        parts = line.split()
        if len(parts) != 5:
            continue
        cls_id = int(parts[0])
        cx, cy, w, h = (float(x) for x in parts[1:])
        rows.append((cls_id, cx, cy, w, h))
    return rows


rows = []
for split in SPLITS:
    for lbl in list_labels(split):
        for cls_id, cx, cy, w, h in parse_yolo_label(lbl):
            rows.append(
                {
                    "split": split,
                    "image": lbl.stem,
                    "cls_id": cls_id,
                    "cls_name": CLASS_NAMES[cls_id],
                    "cx": cx,
                    "cy": cy,
                    "w": w,
                    "h": h,
                    "area": w * h,
                    "aspect": w / h if h > 0 else 0,
                }
            )
ann = pd.DataFrame(rows)
print(f"Total annotations across all splits: {len(ann):,}")
ann.head()

In [ ]:
class_dist = ann.groupby(["split", "cls_name"]).size().unstack(fill_value=0)
class_dist = class_dist.reindex(columns=CLASS_NAMES)
print(class_dist)

fig, ax = plt.subplots(figsize=(8, 4))
class_dist.plot(kind="bar", stacked=False, ax=ax)
ax.set_title("Class distribution per split")
ax.set_ylabel("# bounding boxes")
ax.tick_params(axis="x", rotation=0)
ax.legend(title="class")
plt.tight_layout()
plt.show()

In [ ]:
# Class imbalance ratio (max/min) on train split
train_counts = class_dist.loc["train"]
imbalance = train_counts.max() / max(train_counts.min(), 1)
print("Train counts per class:")
print(train_counts)
print(f"\nImbalance ratio (max/min): {imbalance:.1f}x")
if imbalance > 10:
    print(
        "⚠️  Highly imbalanced — minority classes may be hard to learn without weighted loss / augmentation focus."
    )

## 4. Image-size distribution

In [ ]:
size_rows = []
for split in SPLITS:
    for img_path in list_images(split):
        with Image.open(img_path) as im:
            size_rows.append({"split": split, "width": im.width, "height": im.height})
sizes = pd.DataFrame(size_rows)
print(sizes.groupby("split")[["width", "height"]].agg(["min", "median", "max"]))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col in zip(axes, ["width", "height"], strict=False):
    sns.histplot(sizes, x=col, hue="split", bins=30, ax=ax, multiple="stack")
    ax.set_title(f"Image {col}")
plt.tight_layout()
plt.show()

**Implication for training:** Ultralytics' default `imgsz=640` resizes the longest side to 640 maintaining aspect ratio. If most images are way larger (e.g., 1920×1080) the model can still learn but small objects will lose detail. If they're much smaller than 640 we're upsampling — wasteful.

## 5. Bounding-box size & aspect ratio

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(data=ann, x="cls_name", y="area", ax=axes[0], order=CLASS_NAMES)
axes[0].set_yscale("log")
axes[0].set_title("Normalized bbox area per class (log scale)")
axes[0].set_xlabel("")
axes[0].set_ylabel("width × height (fraction of image)")

sns.boxplot(data=ann, x="cls_name", y="aspect", ax=axes[1], order=CLASS_NAMES)
axes[1].set_title("Aspect ratio (w/h) per class")
axes[1].set_xlabel("")
axes[1].set_ylim(0, 3)

plt.tight_layout()
plt.show()

In [ ]:
# Tiny-object alert: how many boxes are smaller than 0.5% of the image?
tiny = ann[ann["area"] < 0.005]
print(
    f"Tiny boxes (< 0.5% of image area): {len(tiny):,} of {len(ann):,} ({len(tiny)/len(ann)*100:.1f}%)"
)
print("By class:")
print(tiny["cls_name"].value_counts())

**Implication:** the ball typically has a very small bounding box. If a significant fraction of boxes are tiny, we may want:
- Larger `imgsz` (e.g. 960 or 1280) during training to preserve detail
- A model variant tuned for small objects (P2 head)
- Augmentations that preserve small objects (avoid heavy mosaic at high scales)

## 6. Annotations per image

In [ ]:
per_image = ann.groupby(["split", "image"]).size().reset_index(name="n_boxes")
print(per_image.groupby("split")["n_boxes"].describe())

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(per_image, x="n_boxes", hue="split", bins=30, multiple="stack", ax=ax)
ax.set_title("Boxes per image")
plt.tight_layout()
plt.show()

## 7. Visual samples

Random images with bounding boxes overlaid.

In [ ]:
import matplotlib.patches as patches  # noqa: E402

CLASS_COLORS = {
    "ball": "#ff4444",
    "goalkeeper": "#ffaa00",
    "player": "#44ddff",
    "referee": "#aa44ff",
}


def render_sample(image_path: Path, label_path: Path, ax) -> None:
    with Image.open(image_path) as im:
        img_w, img_h = im.size
        ax.imshow(im)
    ax.set_axis_off()
    for cls_id, cx, cy, w, h in parse_yolo_label(label_path):
        x = (cx - w / 2) * img_w
        y = (cy - h / 2) * img_h
        rect = patches.Rectangle(
            (x, y),
            w * img_w,
            h * img_h,
            linewidth=2,
            edgecolor=CLASS_COLORS[CLASS_NAMES[cls_id]],
            facecolor="none",
        )
        ax.add_patch(rect)
        ax.text(
            x,
            max(y - 4, 0),
            CLASS_NAMES[cls_id],
            color="white",
            fontsize=8,
            bbox={"facecolor": CLASS_COLORS[CLASS_NAMES[cls_id]], "pad": 1, "edgecolor": "none"},
        )


sample_imgs = random.sample(list_images("train"), 6)
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, img_path in zip(axes.flat, sample_imgs, strict=False):
    label_path = DATA_ROOT / "train" / "labels" / (img_path.stem + ".txt")
    render_sample(img_path, label_path, ax)
plt.tight_layout()
plt.show()

## 8. Anomaly checks

In [ ]:
issues = []

# Empty label files
for split in SPLITS:
    for lbl in list_labels(split):
        if lbl.stat().st_size == 0:
            issues.append({"split": split, "file": lbl.name, "issue": "empty label file"})

# Missing image for label or label for image
for split in SPLITS:
    img_stems = {p.stem for p in list_images(split)}
    lbl_stems = {p.stem for p in list_labels(split)}
    for orphan in img_stems - lbl_stems:
        issues.append({"split": split, "file": orphan, "issue": "image without label"})
    for orphan in lbl_stems - img_stems:
        issues.append({"split": split, "file": orphan, "issue": "label without image"})

# Boxes outside [0, 1]
oob = ann[
    (ann["cx"] < 0)
    | (ann["cx"] > 1)
    | (ann["cy"] < 0)
    | (ann["cy"] > 1)
    | (ann["w"] <= 0)
    | (ann["w"] > 1)
    | (ann["h"] <= 0)
    | (ann["h"] > 1)
]
for _, r in oob.iterrows():
    issues.append({"split": r["split"], "file": r["image"], "issue": "box out of [0,1]"})

issues_df = pd.DataFrame(issues)
if issues_df.empty:
    print("✓ No anomalies detected.")
else:
    print(f"Found {len(issues_df)} anomalies:")
    print(issues_df["issue"].value_counts())
    issues_df.head(20)

## 9. Conclusions and implications for training

Fill this in **after** running the notebook with your dataset. Typical findings on football-players datasets:

- **Class imbalance is real:** `player` dominates by ~10x over `referee` and `goalkeeper`, and `ball` is the rarest of all. Implication: consider class-weighted loss, focal loss, or oversampling minority classes during data augmentation.
- **Ball is tiny.** Often <0.5% of image area. Consider training at higher `imgsz` (e.g. 960) or using a small-object specialized head (YOLOv11 P2).
- **Image sizes are roughly uniform** (Roboflow often resizes during export). Defaults of `imgsz=640` are reasonable for the baseline.
- **Test split is small** (often ~30-50 images). Don't over-interpret test mAP swings; rely on validation metrics and use test only as a final sanity check.

### Training plan implications
1. **Baseline:** YOLOv11n, `imgsz=640`, `epochs=30`, default augmentations. Quick, gives us a reference.
2. **Improve recall on ball:** later experiment with `imgsz=960` or `1280`.
3. **Improve minority classes:** Optuna HPO sweeping `mosaic`, `mixup`, plus class-weighted training.
4. **Error slicing during analysis:** with FiftyOne, look specifically at where ball detection fails (small? motion-blurred?).

**Next:** integrate MLflow + Hydra into `src/football_tracker/training/train.py` and run the sanity check on MPS, then full baseline on Colab.